# Tạo `pretraining_data_path.txt` cho DocNLC

Notebook này dùng lại script [scripts/create_pretraining_filelist.py](scripts/create_pretraining_filelist.py) để tạo filelist pretrain từ folder `Hybrid/`.

Script Python gốc vẫn được giữ nguyên. Notebook chỉ là giao diện dễ chạy hơn.

## 1. Setup project

In [ ]:
from pathlib import Path
import os
import sys
import subprocess

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "scripts" / "create_pretraining_filelist.py").exists():
    for parent in PROJECT_ROOT.parents:
        if (parent / "scripts" / "create_pretraining_filelist.py").exists():
            PROJECT_ROOT = parent
            break

assert (PROJECT_ROOT / "scripts" / "create_pretraining_filelist.py").exists(), "Không tìm thấy repo DocNLC hoặc script create_pretraining_filelist.py."
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.create_pretraining_filelist import image_files, make_filelist

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Script:", PROJECT_ROOT / "scripts" / "create_pretraining_filelist.py")

## 2. Nhập đường dẫn dataset

Folder cần có cấu trúc:

```text
Hybrid/
├── GT/
└── Degraded/
    ├── Blur/
    ├── Noise/
    ├── Shadow/
    ├── Watermark/
    └── WithBack/
```

In [ ]:
# Sửa đường dẫn này theo máy của bạn.
HYBRID_ROOT = PROJECT_ROOT / "Hybrid"

# File output. Có thể để ngay root repo để pretrain.yml dùng được trực tiếp.
OUTPUT_TXT = PROJECT_ROOT / "pretraining_data_path.txt"

# STRICT=True sẽ báo lỗi nếu thiếu bất kỳ cặp Blur/Noise/Shadow/Watermark nào.
# STRICT=False sẽ bỏ qua group thiếu và in warning.
STRICT = False

HYBRID_ROOT = Path(HYBRID_ROOT).expanduser().resolve()
OUTPUT_TXT = Path(OUTPUT_TXT).expanduser().resolve()

print("HYBRID_ROOT:", HYBRID_ROOT)
print("OUTPUT_TXT:", OUTPUT_TXT)

## 3. Kiểm tra số lượng file trong từng folder

In [ ]:
folders = {
    "GT": HYBRID_ROOT / "GT",
    "WithBack": HYBRID_ROOT / "Degraded" / "WithBack",
    "Blur": HYBRID_ROOT / "Degraded" / "Blur",
    "Noise": HYBRID_ROOT / "Degraded" / "Noise",
    "Shadow": HYBRID_ROOT / "Degraded" / "Shadow",
    "Watermark": HYBRID_ROOT / "Degraded" / "Watermark",
}

for name, folder in folders.items():
    if not folder.is_dir():
        raise FileNotFoundError(f"Không tìm thấy folder {name}: {folder}")

counts = {name: len(image_files(folder)) for name, folder in folders.items()}
for name, count in counts.items():
    print(f"{name:10s}: {count}")

expected_lines = counts["WithBack"] * 3
print("\nSố dòng kỳ vọng nếu đủ bộ 3 variant cho mỗi WithBack:", expected_lines)
print("Blur/Noise/Shadow/Watermark thường nên bằng WithBack * 3.")

## 4. Tạo `pretraining_data_path.txt`

In [ ]:
total_lines, warnings = make_filelist(HYBRID_ROOT, OUTPUT_TXT, strict=STRICT)

print(f"Đã ghi {total_lines} dòng vào: {OUTPUT_TXT}")
if warnings:
    print(f"Có {len(warnings)} group bị bỏ qua vì thiếu match. Một vài ví dụ:")
    for warning in warnings[:20]:
        print(" -", warning)
else:
    print("Không có warning. Dataset match đầy đủ.")

## 5. Preview file output

In [ ]:
lines = [line.strip() for line in OUTPUT_TXT.read_text().splitlines() if line.strip()]
print("Tổng số dòng:", len(lines))
print("\n3 dòng đầu:")
for line in lines[:3]:
    print(line)

print("\nKiểm tra format 6 cột:")
bad_lines = [(idx, line) for idx, line in enumerate(lines, 1) if len(line.split("|")) != 6]
if bad_lines:
    print("Có dòng sai format. Ví dụ:", bad_lines[:3])
else:
    print("OK: tất cả dòng có đúng 6 cột.")

## 6. Dùng file này cho pretrain

Trong `options/pretrain.yml` hoặc notebook pretrain, trỏ train filelist tới file vừa tạo:

```yaml
datasets:
  train:
    mode: multi_task
    filelist: /path/to/pretraining_data_path.txt
```

Mỗi dòng có format:

```text
GT|WithBack|Blur|Noise|Shadow|Watermark
```